# 🗒️ Meeting Assistant RL Environment — Demo

This notebook demonstrates the **Meeting Assistant OpenEnv environment**.

An agent is given raw meeting notes and must produce:
- A concise **summary**
- A list of **action items**

Reward (0–1) is computed from keyword overlap with reference outputs.

---

## 1. Install dependencies

In [ ]:
!pip install openenv-core transformers torch --quiet

## 2. Run the environment locally (no Docker needed for demo)

In [ ]:
import sys
sys.path.insert(0, '..')  # adjust if running from notebooks/

from meeting_assistant_env.models import MeetingAction
from meeting_assistant_env.server.meeting_environment import MeetingEnvironment

env = MeetingEnvironment(max_turns=3)
obs = env.reset()

print('📋 Meeting Notes:')
print(obs.meeting_notes)
print(f'\nTurn {obs.turn}/{obs.max_turns}')

## 3. Agent 1 — Naive rule-based agent (baseline)

In [ ]:
def naive_agent(meeting_notes: str) -> MeetingAction:
    """Simple keyword-based agent — your original hackathon code!"""
    sentences = [s.strip() for s in meeting_notes.split('.') if s.strip()]
    summary = '. '.join(sentences[:2]) + '.'

    actions = []
    keywords = ['will', 'needs to', 'to fix', 'to complete', 'by friday',
                 'by monday', 'by wednesday', 'follow-up', 'to send', 'to deploy']
    for sentence in sentences:
        if any(kw in sentence.lower() for kw in keywords):
            actions.append(sentence.strip())

    return MeetingAction(summary=summary, action_items=actions)

action = naive_agent(obs.meeting_notes)
result = env.step(action)

print('🤖 Agent summary:', action.summary)
print('✅ Action items:', action.action_items)
print(f'\n🏆 Reward: {result.reward:.3f}')
print(f'   Summary score: {result.info["summary_score"]}')
print(f'   Action item score: {result.info["action_item_score"]}')

## 4. Agent 2 — Transformer-powered agent (your original model!)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained('sshleifer/distilbart-cnn-12-6')
model = AutoModelForSeq2SeqLM.from_pretrained('sshleifer/distilbart-cnn-12-6')

def transformer_agent(meeting_notes: str) -> MeetingAction:
    """Transformer-based agent using DistilBART for summarisation."""
    inputs = tokenizer(
        meeting_notes, max_length=1024, return_tensors='pt', truncation=True
    )
    summary_ids = model.generate(
        inputs['input_ids'],
        max_new_length=min(50, len(meeting_notes.split())),
        min_length=10,
        early_stopping=True,
    )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True,
                               clean_up_tokenization_spaces=True)

    # Keyword-based action extraction (upgraded from original)
    action_keywords = [
        'will', 'needs to', 'to fix', 'to complete', 'to deploy',
        'to send', 'to review', 'by friday', 'by monday', 'by wednesday',
        'follow-up', 'scheduled', 'postponed', 'assigned'
    ]
    sentences = [s.strip() for s in meeting_notes.split('.') if s.strip()]
    action_items = [
        s for s in sentences
        if any(kw in s.lower() for kw in action_keywords)
    ]

    return MeetingAction(summary=summary, action_items=action_items)

# Reset for a fresh episode
obs = env.reset()
print('📋 Meeting Notes:', obs.meeting_notes)

action = transformer_agent(obs.meeting_notes)
result = env.step(action)

print('\n🤖 Transformer summary:', action.summary)
print('✅ Action items:', action.action_items)
print(f'\n🏆 Reward: {result.reward:.3f}')
print(f'   Summary score: {result.info["summary_score"]}')
print(f'   Action item score: {result.info["action_item_score"]}')

## 5. Run a full episode (3 turns)
Show how reward evolves across multiple turns.

In [ ]:
obs = env.reset()
total_reward = 0.0

print(f'Episode started | Meeting notes:\n{obs.meeting_notes}\n')

while True:
    action = naive_agent(obs.meeting_notes)
    result = env.step(action)
    total_reward += result.reward

    print(f'Turn {result.observation.turn} | reward={result.reward:.3f} | done={result.done}')

    if result.done:
        break
    obs = result.observation

print(f'\n📊 Total episode reward: {total_reward:.3f}')